# 02 | Sales generation

Generates the daily sales of every store and product with a **multiplicative model**: the expected number of units sold is the base sales multiplied by a set of simple factors, each one representing a real pattern.

```
expected units = product base × store base × season × geography × tourism
                 × weekday and opening days × special dates × trend
                 × store life cycle × product life cycle × promotion
```

The quantity actually sold is then drawn from a **Poisson distribution** with that average, which adds realistic random variation (some days above, others below, many days with zero sales of expensive products).

| Factor | Example |
|---|---|
| Store base | A 2 000 m² shopping centre in a large city sells more than a 500 m² street store in a rural area |
| Season | Swimsuits in July, ski jackets in December, running shoes in January |
| Geography | Coastal stores sell more beach products; mountain stores sell more snow products |
| Tourism | Coastal stores sell 20% more in July and August |
| Weekday | Saturday is the busiest day; street stores close on Sundays and public holidays |
| Special dates | Black Friday weekend and the weeks before Christmas |
| Trend | The chain grows 4% a year; each store has its own growth (between −3% and +4%) |
| Store life cycle | New stores need six months to reach their full sales |
| Product life cycle | Launched products only sell after their launch; the e-bike grows 25% a year |
| Promotion | Each 10% of discount increases the units sold by 15% |

## Setup

In [1]:
import sys
from pathlib import Path

# Make the src/ folder importable when running from the notebooks/ folder
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.config import *
from src.utils import *

rng = np.random.default_rng(SEED)

dim_store = load_table("dim_store")
dim_product = load_table("dim_product")
dim_date = load_table("dim_date")
dim_promotion = load_table("dim_promotion")

n_days, n_stores, n_products = len(dim_date), len(dim_store), len(dim_product)
print(f"{n_days} days × {n_stores} stores × {n_products} products = {n_days * n_stores * n_products:,} combinations")

1826 days × 52 stores × 33 products = 3,133,416 combinations


## Model parameters

All the numbers that shape the data are in this cell, so they are easy to find and change.

In [2]:
# Store base
STORE_TYPE_FACTOR = {"shopping_centre": 1.15, "street": 1.0, "retail_park": 0.9}
POPULATION_TIER_FACTOR = {1: 0.7, 2: 0.85, 3: 1.0, 4: 1.15, 5: 1.3}

# Season: monthly factor by category (January ... December)
SEASONALITY = {
    "Running":          [1.4, 1.0, 1.1, 1.2, 1.2, 1.0, 0.8, 0.7, 1.1, 1.1, 1.0, 1.1],
    "Cycling":          [0.6, 0.7, 1.0, 1.3, 1.4, 1.4, 1.3, 1.1, 1.0, 0.8, 0.7, 1.0],
    "Hiking & Camping": [0.5, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.5, 1.0, 0.9, 0.7, 0.8],
    "Beach & Water":    [0.2, 0.2, 0.4, 0.7, 1.2, 2.0, 2.6, 2.4, 1.0, 0.4, 0.2, 0.3],
    "Snow & Winter":    [2.2, 1.8, 0.9, 0.3, 0.1, 0.1, 0.1, 0.1, 0.3, 0.8, 1.8, 2.4],
    "Fitness":          [1.8, 1.3, 1.1, 1.0, 0.9, 0.8, 0.7, 0.7, 1.4, 1.1, 1.0, 0.9],
}

# Geography: factor by (location profile, category); any combination not listed = 1
LOCATION_FACTOR = {
    ("Coastal", "Beach & Water"): 1.5,
    ("Inland", "Beach & Water"): 0.4,
    ("Mountain", "Beach & Water"): 0.4,
    ("Mountain", "Snow & Winter"): 3.0,
    ("Coastal", "Snow & Winter"): 0.5,
    ("Inland", "Snow & Winter"): 0.6,
    ("Mountain", "Hiking & Camping"): 1.4,
}
URBAN_FACTOR = {
    ("City", "Running"): 1.2, ("City", "Fitness"): 1.2,
    ("Rural", "Running"): 0.8, ("Rural", "Fitness"): 0.8,
}
TOURISM_FACTOR = 1.2            # coastal stores in July and August

# Weekday profile by store type (Monday ... Sunday); 0 = closed
WEEKDAY_FACTOR = {
    "street":          [0.80, 0.85, 0.90, 0.95, 1.20, 1.70, 0.00],
    "shopping_centre": [0.80, 0.80, 0.85, 0.90, 1.15, 1.60, 1.30],
    "retail_park":     [0.75, 0.80, 0.85, 0.90, 1.10, 1.70, 1.00],
}

# Special dates
BLACK_FRIDAY_FACTOR = [2.5, 1.6, 1.4, 1.3]   # Friday, Saturday, Sunday, Monday
CHRISTMAS_FACTOR = 1.4                       # 1 to 23 December
CHRISTMAS_EVE_FACTOR = 0.8                   # 24 December

# Trend
CHAIN_GROWTH = 0.04                          # +4% a year for the whole chain
STORE_GROWTH_RANGE = (-0.03, 0.04)           # extra growth of each store, chosen at random
PRODUCT_GROWTH = {"E-bike": 0.25, "Stand-up paddle board": 0.20, "GPS sports watch": 0.15}

# Life cycle of new stores
RAMP_UP_DAYS = 180
RAMP_UP_START = 0.6                          # a new store starts at 60% of its normal sales

# Promotions
PROMOTION_EFFECT = 1.5                       # units × (1 + 1.5 × discount): 20% off -> +30% units

# Prices (fictitious inflation index and country price level)
PRICE_INDEX = {2021: 1.00, 2022: 1.04, 2023: 1.09, 2024: 1.12, 2025: 1.14}
COUNTRY_PRICE_FACTOR = {"PT": 1.00, "ES": 1.03}

## Building the factors

Each factor is an array with one value per day, store or product (or a combination). In the end they are all multiplied together.

In [3]:
dates = pd.to_datetime(dim_date["date"])
month_index = dim_date["month"].values - 1                      # 0 = January
weekday_index = dim_date["weekday"].values - 1                  # 0 = Monday
years_since_start = (dates - dates.iloc[0]).dt.days.values / 365.25
categories = dim_product["category"].values

### 1. Product base and store base

In [4]:
product_base = dim_product["base_daily_units"].values                              # (products)

store_base = (
    dim_store["size_m2"] / 1000
    * dim_store["store_type"].map(STORE_TYPE_FACTOR)
    * dim_store["population_tier"].map(POPULATION_TIER_FACTOR)
).values                                                                           # (stores)

dim_store.assign(store_base=store_base.round(2))[["store_name", "store_type", "size_m2", "population_tier", "store_base"]].head()

,store_name,store_type,size_m2,population_tier,store_base
0,Cume & Costa São Bernardo,shopping_centre,1650,5,2.47
1,Cume & Costa Campolide,shopping_centre,1550,5,2.32
2,Cume & Costa Boticas e Granja,street,400,4,0.46
3,Cume & Costa Esmoriz,shopping_centre,1250,5,1.87
4,Cume & Costa Antuzede e Vil de Matos,street,600,4,0.69


### 2. Season (day × product)

In [5]:
season_by_month = np.array([SEASONALITY[category] for category in categories]).T   # (12 months, products)
season = season_by_month[month_index]                                                    # (days, products)

### 3. Geography (store × product) and tourism (day × store)

In [6]:
geography = np.ones((n_stores, n_products))
for s, store in dim_store.iterrows():
    for p, category in enumerate(categories):
        geography[s, p] = (LOCATION_FACTOR.get((store["location_profile"], category), 1.0)
                           * URBAN_FACTOR.get((store["urban_level"], category), 1.0))

is_coastal = (dim_store["location_profile"] == "Coastal").values
is_summer_peak = np.isin(dim_date["month"].values, [7, 8])
tourism = np.where(is_summer_peak[:, None] & is_coastal[None, :], TOURISM_FACTOR, 1.0)   # (days, stores)

### 4. Weekday and opening days (day × store)

* Street stores close on Sundays and on the national public holidays of their country.
* Every store closes on 25 December and 1 January.

In [7]:
weekday = np.array([WEEKDAY_FACTOR[store_type] for store_type in dim_store["store_type"]]).T   # (7, stores)
weekday = weekday[weekday_index]                                                                 # (days, stores)

is_holiday = np.where(
    (dim_store["country"] == "PT").values[None, :],
    dim_date["is_holiday_pt"].values[:, None],
    dim_date["is_holiday_es"].values[:, None],
)
is_street = (dim_store["store_type"] == "street").values[None, :]
weekday = np.where(is_holiday & is_street, 0.0, weekday)

closed_for_everyone = (dim_date["month"].values == 12) & (dates.dt.day.values == 25) \
                    | (dim_date["month"].values == 1) & (dates.dt.day.values == 1)
weekday[closed_for_everyone, :] = 0.0

### 5. Special dates (day)

In [8]:
special = np.ones(n_days)

is_december = dim_date["month"].values == 12
day_of_month = dates.dt.day.values
special[is_december & (day_of_month <= 23)] = CHRISTMAS_FACTOR
special[is_december & (day_of_month == 24)] = CHRISTMAS_EVE_FACTOR

for promotion in dim_promotion[dim_promotion["promotion_name"] == "Black Friday"].itertuples():
    for offset, factor in enumerate(BLACK_FRIDAY_FACTOR):
        special[(dates == promotion.start_date + pd.Timedelta(days=offset)).values] = factor

### 6. Trend (day × store)

In [9]:
store_growth = rng.uniform(*STORE_GROWTH_RANGE, size=n_stores)

# Straight-line growth: after 2 years at +4% a year, the factor is 1 + 0.04 × 2 = 1.08
trend = (1 + CHAIN_GROWTH * years_since_start)[:, None] * (1 + store_growth[None, :] * years_since_start[:, None])

### 7. Store life cycle (day × store)

0 before opening and after closing; in the first 180 days, sales grow in a straight line from 60% to 100%.

In [10]:
opening = pd.to_datetime(dim_store["opening_date"]).values
closing = pd.to_datetime(dim_store["closing_date"]).fillna(pd.Timestamp("2100-01-01")).values

days_open = (dates.values[:, None] - opening[None, :]) / np.timedelta64(1, "D")       # (days, stores)
store_life = RAMP_UP_START + (1 - RAMP_UP_START) * days_open / RAMP_UP_DAYS
store_life = np.clip(store_life, None, 1.0)
store_life[days_open < 0] = 0.0                                                       # not open yet
store_life[dates.values[:, None] > closing[None, :]] = 0.0                            # already closed

### 8. Product life cycle (day × product)

In [11]:
launch = pd.to_datetime(dim_product["launch_date"]).fillna(dates.iloc[0]).values
discontinued = pd.to_datetime(dim_product["discontinued_date"]).fillna(pd.Timestamp("2100-01-01")).values

is_available = (dates.values[:, None] >= launch[None, :]) & (dates.values[:, None] <= discontinued[None, :])

years_since_launch = np.clip((dates.values[:, None] - launch[None, :]) / np.timedelta64(1, "D") / 365.25, 0, None)
product_growth = dim_product["product_name"].map(PRODUCT_GROWTH).fillna(0).values
product_life = np.where(is_available, 1 + product_growth[None, :] * years_since_launch, 0.0)

### 9. Promotions (day × product)

If two promotions overlap for the same product, the larger discount is used.

In [12]:
discount = np.zeros((n_days, n_products))
promotion_id = np.full((n_days, n_products), None, dtype=object)

for promotion in dim_promotion.itertuples():
    in_period = ((dates >= promotion.start_date) & (dates <= promotion.end_date)).values
    in_categories = np.isin(categories, [c.strip() for c in promotion.categories.split(",")])
    applies = in_period[:, None] & in_categories[None, :] & (promotion.discount_pct > discount)
    discount[applies] = promotion.discount_pct
    promotion_id[applies] = promotion.promotion_id

promotion_effect = 1 + PROMOTION_EFFECT * discount

## Expected units and simulated sales

In [13]:
expected_units = (
    product_base[None, None, :]
    * store_base[None, :, None]
    * season[:, None, :]
    * geography[None, :, :]
    * tourism[:, :, None]
    * weekday[:, :, None]
    * special[:, None, None]
    * trend[:, :, None]
    * store_life[:, :, None]
    * product_life[:, None, :]
    * promotion_effect[:, None, :]
)                                                       # (days, stores, products)

quantity = rng.poisson(expected_units)
print(f"Expected units: {expected_units.sum():,.0f} | Simulated units: {quantity.sum():,}")

Expected units: 8,179,898 | Simulated units: 8,182,791


## Fact table (`fact_sales`)

Only the combinations with at least one unit sold are kept (a real sales table has no rows for products that were not sold).

* `unit_price` = base price × price index of the year × country price level
* `revenue` = quantity × unit price × (1 − discount)
* `cost` = quantity × unit cost × price index of the year

In [14]:
d, s, p = np.nonzero(quantity)

fact_sales = pd.DataFrame({
    "date": dates.values[d],
    "store_id": dim_store["store_id"].values[s],
    "product_id": dim_product["product_id"].values[p],
    "quantity": quantity[d, s, p],
    "promotion_id": promotion_id[d, p],
    "discount_pct": discount[d, p],
})

price_index = dim_date["year"].map(PRICE_INDEX).values[d]
country_price = dim_store["country"].map(COUNTRY_PRICE_FACTOR).values[s]

fact_sales["unit_price"] = (dim_product["base_price"].values[p] * price_index * country_price).round(2)
fact_sales["revenue"] = (fact_sales["quantity"] * fact_sales["unit_price"] * (1 - fact_sales["discount_pct"])).round(2)
fact_sales["cost"] = (fact_sales["quantity"] * dim_product["unit_cost"].values[p] * price_index).round(2)

fact_sales = fact_sales.sort_values(["date", "store_id", "product_id"]).reset_index(drop=True)
print(f"{len(fact_sales):,} rows")
fact_sales.head()

1,504,031 rows


,date,store_id,product_id,quantity,promotion_id,discount_pct,unit_price,revenue,cost
0,2021-01-02,S001,P001,24,NaN,0.0,89.99,2159.76,1190.64
1,2021-01-02,S001,P002,14,NaN,0.0,109.99,1539.86,906.78
2,2021-01-02,S001,P003,26,NaN,0.0,19.99,519.74,294.32
3,2021-01-02,S001,P004,14,NaN,0.0,17.99,251.86,123.34
4,2021-01-02,S001,P005,36,NaN,0.0,9.99,359.64,219.60


## Quick check

In [15]:
summary = (
    fact_sales.assign(year=fact_sales["date"].dt.year)
    .groupby("year")
    .agg(units=("quantity", "sum"), revenue=("revenue", "sum"), cost=("cost", "sum"))
)
summary["margin_%"] = (100 * (summary["revenue"] - summary["cost"]) / summary["revenue"]).round(1)
summary["revenue_growth_%"] = (100 * summary["revenue"].pct_change()).round(1)
summary.style.format({"units": "{:,.0f}", "revenue": "€{:,.0f}", "cost": "€{:,.0f}"})

,units,revenue,cost,margin_%,revenue_growth_%
year,,,,,
2021,"1,291,713","€54,039,295","€30,830,586",42.900000,nan
2022,"1,448,815","€66,115,926","€37,815,115",42.800000,22.300000
2023,"1,691,236","€83,751,599","€47,879,819",42.800000,26.700000
2024,"1,802,376","€92,737,196","€53,103,046",42.700000,10.700000
2025,"1,948,651","€103,122,866","€59,052,981",42.700000,11.200000


In [16]:
# Sanity check: average annual revenue per store in the last year, by store type
# (only stores open for the whole year)
last_year = dim_date["year"].max()
full_year_stores = dim_store[(dim_store["opening_date"] < f"{last_year}-01-01") & dim_store["closing_date"].isna()]

last_year_revenue = (
    fact_sales[(fact_sales["date"].dt.year == last_year) & fact_sales["store_id"].isin(full_year_stores["store_id"])]
    .groupby("store_id")["revenue"].sum()
    .rename("revenue")
    .reset_index()
    .merge(dim_store[["store_id", "store_type"]], on="store_id")
)
last_year_revenue.groupby("store_type")["revenue"].agg(["count", "mean", "min", "max"]).style.format("€{:,.0f}", subset=["mean", "min", "max"])

,count,mean,min,max
store_type,,,,
retail_park,13,"€3,325,260","€1,581,388","€4,798,669"
shopping_centre,15,"€3,042,555","€2,185,314","€3,982,765"
street,18,"€605,024","€252,787","€870,807"


## Save

In [17]:
save_table(fact_sales, "fact_sales")

Saved 'fact_sales' (1,504,031 rows)


## Summary
* Built ten factors, each one with simple multiplications, representing the store size and location, the season, geography, tourism, opening days, special dates, growth, store and product life cycles and promotions.
* Simulated the daily units with a Poisson distribution and calculated prices, revenue and cost.
* The fixed random seed (`SEED` in `src/config.py`) makes the data exactly reproducible.